In [1]:
import random
import json

#### JSON from the editor
```json
{
  "nodes": [
    {
      "width": 240,
      "height": 276,
      "id": "0.05272377425322505-1756500835216",
      "type": "eventType",
      "position": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "data": {
        "box": {
          "newEventVars": {
            "0": [
              "payment reminder"
            ],
            "1": [
              "pay order"
            ]
          },
          "newObjectVars": {
            "0": [
              "orders"
            ]
          },
          "filters": [
            {
              "type": "O2E",
              "object": 0,
              "event": 0,
              "qualifier": null
            },
            {
              "type": "O2E",
              "object": 0,
              "event": 1,
              "qualifier": null
            }
          ],
          "sizeFilters": [],
          "constraints": [
            {
              "type": "Filter",
              "filter": {
                "type": "TimeBetweenEvents",
                "from_event": 0,
                "to_event": 1,
                "min_seconds": 0,
                "max_seconds": null
              }
            }
          ],
          "evVarLabels": {},
          "obVarLabels": {}
        }
      },
      "positionAbsolute": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "selected": true,
      "dragging": false
    }
  ],
  "edges": []
}
```

#### JSON given to the DB
```json
{
  "tree": {
    "nodes": [
      {
         "Box": [
           {
             "newEventVars": { "0": ["payment reminder"], "1": ["pay order"] },
             "newObjectVars": { "0": ["orders"] },
             "filters": [
               { "type": "O2E", "object": 0, "event": 0, "qualifier": null },
               { "type": "O2E", "object": 0, "event": 1, "qualifier": null }
             ],
             "sizeFilters": [],
             "constraints": [
               {
                 "type": "Filter",
                 "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": null }
               }
             ],
             "evVarLabels": {},
             "obVarLabels": {}
           },
           [] // IDs of any connected child nodes in the tree
         ]
      }
    ],
    "edgeNames": []
  },
  "database": "DuckDB", // Target Database Engine (if doing DB translation)
  "table_mappings": { ... }
}
```

In [2]:
patterns = [
    "EVENT_A happens before EVENT_B",
    "Whenever EVENT_A occurs, EVENT_B eventually occurs",
    "After EVENT_A, EVENT_B must not occur",
    "EVENT_A and EVENT_B refer to the same OBJECT",
    "EVENT_B happens within TIME hours after EVENT_A"
]

events = [
    "create order",
    "add item",
    "ship package",
    "pay order",
    "cancel order",
    "approve application",
    "submit application",
    "review application",
    "merge pull request",
    "open issue"
]

objects = [
    "orders",
    "items",
    "applications",
    "pull requests",
    "issues"
]

In [3]:
def generate_example():
    pattern = random.choice(patterns)

    A = random.choice(events)
    B = random.choice(events)
    obj = random.choice(objects)
    t = random.choice([1, 6, 24, 72])

    text = pattern.replace("EVENT_A", A)\
                  .replace("EVENT_B", B)\
                  .replace("OBJECT", obj)\
                  .replace("TIME", str(t))

    ocpq = {
        "pattern": pattern,
        "event_A": A,
        "event_B": B,
        "object": obj,
        "time": t
    }

    return {"nl": text, "query": ocpq}

dataset = [generate_example() for _ in range(200)]

# with open("ocpq_dataset.json","w") as f:
#     json.dump(dataset,f,indent=2)

In [4]:
dataset[:5]

[{'nl': 'review application happens before merge pull request',
  'query': {'pattern': 'EVENT_A happens before EVENT_B',
   'event_A': 'review application',
   'event_B': 'merge pull request',
   'object': 'applications',
   'time': 1}},
 {'nl': 'pay order happens before ship package',
  'query': {'pattern': 'EVENT_A happens before EVENT_B',
   'event_A': 'pay order',
   'event_B': 'ship package',
   'object': 'issues',
   'time': 24}},
 {'nl': 'After approve application, approve application must not occur',
  'query': {'pattern': 'After EVENT_A, EVENT_B must not occur',
   'event_A': 'approve application',
   'event_B': 'approve application',
   'object': 'pull requests',
   'time': 1}},
 {'nl': 'Whenever add item occurs, cancel order eventually occurs',
  'query': {'pattern': 'Whenever EVENT_A occurs, EVENT_B eventually occurs',
   'event_A': 'add item',
   'event_B': 'cancel order',
   'object': 'pull requests',
   'time': 6}},
 {'nl': 'Whenever approve application occurs, submit ap